# 📰 Fake News Detection Pipeline
### Expert Machine Learning Implementation

This notebook implements a modular, high-performance pipeline to classify news articles as **Real** or **Fake**. 

---

## 1. Imports & Setup
First, we import the necessary libraries and ensure that our environment is ready for NLP tasks.

In [ ]:
import os
import re
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Ensure NLTK resources are available
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

if not os.path.exists('submissions'):
    os.makedirs('submissions')
    print("Created 'submissions' directory.")

## 2. Data Loading
We load our training and testing datasets. If you are using your own data, ensure the CSV files are correctly named.

In [ ]:
def load_data(train_path='xy_train.csv', test_path='x_test.csv'):
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print(f"Loaded {len(train_df)} training samples and {len(test_df)} testing samples.")
    return train_df, test_df

try:
    train_df, test_df = load_data()
    display(train_df.head())
except FileNotFoundError:
    print("Error: Data files not found. Please run the dummy data generator first.")

## 3. Text Preprocessing
We clean the text data using Regex to remove noise and NLTK for tokenization and stopword removal.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # Lowercase conversion
    text = text.lower()
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenization
    tokens = word_tokenize(text)
    
    # Stopword removal
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [w for w in tokens if w not in stop_words]
    
    return " ".join(filtered_tokens)

# Apply cleaning
text_col = 'text' if 'text' in train_df.columns else train_df.columns[0]
train_df['cleaned_text'] = train_df[text_col].apply(clean_text)
test_df['cleaned_text'] = test_df[text_col].apply(clean_text)

print("Preprocessing complete.")
display(train_df[['text', 'cleaned_text']].head())

## 4. Feature Extraction
Converting text to numerical vectors using TF-IDF and splitting the data.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
label_col = 'label' if 'label' in train_df.columns else train_df.columns[-1]

X = vectorizer.fit_transform(train_df['cleaned_text'])
y = train_df[label_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_test_tfidf = vectorizer.transform(test_df['cleaned_text'])

print(f"Feature extraction complete. Training shape: {X_train.shape}")

## 5. Model Training & Evaluation
We iterate through 8 different classifiers to find the best performing model.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Multinomial NB': MultinomialNB(),
    'XGBoost': xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1),
    'CatBoost': CatBoostClassifier(verbose=0, random_state=42)
}

trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)
    
    print(f"{name} Accuracy: {accuracy_score(y_val, y_pred):.4f}")
    print(classification_report(y_val, y_pred))
    trained_models[name] = model

## 6. Prediction & Export
Finally, we generate submission files for each model.

In [ ]:
for name, model in trained_models.items():
    short_name = name.split()[0].replace("Logistic", "Logis").replace("Random", "RF")
    filename = f"submissions/{short_name}_submission.csv"
    
    preds = model.predict(X_test_tfidf)
    sub_df = pd.DataFrame({'id': test_df.index, 'label': preds})
    sub_df.to_csv(filename, index=False)
    print(f"Saved: {filename}")